In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load data
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print("Data loaded!")
print(f"X_train shape: {X_train.shape}")

Data loaded!
X_train shape: (1166, 86)


In [2]:
# Cross Validation
ridge = Ridge(alpha=1.0)

cv_scores = cross_val_score(
    ridge, X_train, y_train,
    cv=5,
    scoring='r2'
)

print("Cross Validation Complete!")
print(f"\nCV Scores per fold:")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: {score:.4f}")

print(f"\nMean CV Score: {cv_scores.mean():.4f}")
print(f"Std CV Score:  {cv_scores.std():.4f}")
print(f"\nConsistent scores = Model is stable & reliable!")

Cross Validation Complete!

CV Scores per fold:
  Fold 1: 0.8963
  Fold 2: 0.8740
  Fold 3: 0.8884
  Fold 4: 0.9096
  Fold 5: 0.9099

Mean CV Score: 0.8956
Std CV Score:  0.0136

Consistent scores = Model is stable & reliable!


In [3]:
# Hyperparameter Tuning with GridSearchCV
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 5.0, 
              10.0, 50.0, 100.0, 500.0, 1000.0]
}

grid_search = GridSearchCV(
    Ridge(),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Hyperparameter Tuning Complete!")
print(f"\nBest Alpha: {grid_search.best_params_['alpha']}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Hyperparameter Tuning Complete!

Best Alpha: 100.0
Best CV Score: 0.8982


In [4]:
# Evaluate best model on test set
best_model = grid_search.best_estimator_
preds = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("Tuned Model Evaluation:")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# Convert predictions back to actual prices
actual_prices = np.expm1(y_test)
predicted_prices = np.expm1(preds)

print(f"\nSample Predictions vs Actual:")
print(f"{'Actual':>12} {'Predicted':>12} {'Difference':>12}")
print("-" * 40)
for actual, pred in zip(actual_prices[:5], predicted_prices[:5]):
    diff = abs(actual - pred)
    print(f"${actual:>11,.0f} ${pred:>11,.0f} ${diff:>11,.0f}")

Tuned Model Evaluation:
RMSE: 0.0029
R²:   0.8925

Sample Predictions vs Actual:
      Actual    Predicted   Difference
----------------------------------------
$          3 $          3 $          0
$          3 $          3 $          0
$          3 $          3 $          0
$          3 $          3 $          0
$          3 $          3 $          0


In [5]:
# Check what y_test looks like
print("y_test sample values:")
print(y_test[:5])
print(f"\ny_test min: {y_test.min()}")
print(f"y_test max: {y_test.max()}")
print(f"\nAfter expm1:")
print(np.expm1(y_test[:5]))

y_test sample values:
0    1.274465
1    1.260380
2    1.263525
3    1.270646
4    1.284985
Name: SalePrice, dtype: float64

y_test min: 1.2354445458320145
y_test max: 1.2921429199412908

After expm1:
0    2.576786
1    2.526763
2    2.537870
3    2.563153
4    2.614612
Name: SalePrice, dtype: float64


In [6]:
# Load original cleaned data and get correct y values
df_clean = pd.read_csv('../data/train_cleaned.csv')

print("SalePrice sample from cleaned data:")
print(df_clean['SalePrice'][:5])
print(f"\nMin: {df_clean['SalePrice'].min()}")
print(f"Max: {df_clean['SalePrice'].max()}")

SalePrice sample from cleaned data:
0    1.276430
1    1.273490
2    1.277889
3    1.267876
4    1.280221
Name: SalePrice, dtype: float64

Min: 1.2351476799089802
Max: 1.3019675689821726


In [7]:
import numpy as np
import pandas as pd

# Reload from original
df_original = pd.read_csv('../data/train.csv')

print("Original SalePrice sample:")
print(df_original['SalePrice'][:5])
print(f"\nMin: {df_original['SalePrice'].min()}")
print(f"Max: {df_original['SalePrice'].max()}")

# Apply correct log transform
df_original['SalePrice'] = np.log1p(df_original['SalePrice'])

print("\nAfter correct log transform:")
print(df_original['SalePrice'][:5])
print(f"\nMin: {df_original['SalePrice'].min()}")
print(f"Max: {df_original['SalePrice'].max()}")

Original SalePrice sample:
0    208500
1    181500
2    223500
3    140000
4    250000
Name: SalePrice, dtype: int64

Min: 34900
Max: 755000

After correct log transform:
0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64

Min: 10.460270761075149
Max: 13.534474352733596


In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Step 1 - Load original data
df = pd.read_csv('../data/train.csv')
print(f"Original shape: {df.shape}")

# Step 2 - Remove outliers
df = df.drop(df[(df['GrLivArea'] > 4000) & 
                (df['SalePrice'] < 200000)].index)
print(f"After outlier removal: {df.shape}")

# Step 3 - Correct log transform on SalePrice
df['SalePrice'] = np.log1p(df['SalePrice'])
print(f"SalePrice range: {df['SalePrice'].min():.2f} to {df['SalePrice'].max():.2f}")

# Step 4 - Handle missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Step 5 - Encode categorical
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = pd.factorize(df[col])[0]

# Step 6 - Feature Engineering
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
df['TotalBath'] = (df['FullBath'] + df['HalfBath'] * 0.5 +
                   df['BsmtFullBath'] + df['BsmtHalfBath'] * 0.5)
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
df['HasPool'] = (df['PoolArea'] > 0).astype(int)

# Step 7 - Separate X and y
X = df.drop(columns=['SalePrice', 'Id'])
y = df['SalePrice']

# Step 8 - Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Step 9 - Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Step 10 - Save everything
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
joblib.dump(scaler, '../models/scaler.pkl')

print("\nEverything fixed and saved correctly!")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train range: {y_train.min():.2f} to {y_train.max():.2f}")
print(f"y_test range:  {y_test.min():.2f} to {y_test.max():.2f}")

Original shape: (1460, 81)
After outlier removal: (1458, 81)
SalePrice range: 10.46 to 13.53


TypeError: Cannot perform reduction 'median' with string dtype

In [10]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train min: {y_train.min():.2f}")
print(f"y_train max: {y_train.max():.2f}")
print(f"y_test min:  {y_test.min():.2f}")
print(f"y_test max:  {y_test.max():.2f}")

X_train shape: (1166, 86)
X_test shape:  (292, 86)
y_train min: 1.24
y_train max: 1.30
y_test min:  1.24
y_test max:  1.29


In [11]:
# Check what y looks like before saving
print("y sample before saving:")
print(y[:5])
print(f"y min: {y.min():.2f}")
print(f"y max: {y.max():.2f}")

y sample before saving:
0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64
y min: 10.46
y max: 13.53


In [12]:
# Save y correctly
y_train.to_frame().to_csv('../data/y_train.csv', index=False)
y_test.to_frame().to_csv('../data/y_test.csv', index=False)

# Reload and verify
y_train_check = pd.read_csv('../data/y_train.csv').squeeze()
y_test_check = pd.read_csv('../data/y_test.csv').squeeze()

print("y saved and reloaded!")
print(f"\ny_train sample:")
print(y_train_check[:5])
print(f"\ny_train min: {y_train_check.min():.2f}")
print(f"y_train max: {y_train_check.max():.2f}")
print(f"\ny_test min: {y_test_check.min():.2f}")
print(f"y_test max: {y_test_check.max():.2f}")

y saved and reloaded!

y_train sample:
0    1.268643
1    1.273074
2    1.269163
3    1.272710
4    1.265731
Name: SalePrice, dtype: float64

y_train min: 1.24
y_train max: 1.30

y_test min: 1.24
y_test max: 1.29


In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Step 1 - Load original data
df = pd.read_csv('../data/train.csv')

# Step 2 - Remove outliers
df = df.drop(df[(df['GrLivArea'] > 4000) & 
                (df['SalePrice'] < 200000)].index)

# Step 3 - Correct log transform
y = np.log1p(df['SalePrice'].values)
print(f"y range: {y.min():.2f} to {y.max():.2f}")

# Step 4 - Save y directly as numpy
np.save('../data/y_train_raw.npy', y)

# Step 5 - Split y
from sklearn.model_selection import train_test_split
indices = np.arange(len(y))
idx_train, idx_test = train_test_split(
    indices, test_size=0.2, random_state=42
)

y_train_correct = y[idx_train]
y_test_correct = y[idx_test]

np.save('../data/y_train.npy', y_train_correct)
np.save('../data/y_test.npy', y_test_correct)

print(f"y saved as numpy files!")
print(f"y_train range: {y_train_correct.min():.2f} to {y_train_correct.max():.2f}")
print(f"y_test range:  {y_test_correct.min():.2f} to {y_test_correct.max():.2f}")

y range: 10.46 to 13.53
y saved as numpy files!
y_train range: 10.46 to 13.53
y_test range:  10.47 to 13.02


In [18]:
# Load correct y
y_train_correct = np.load('../data/y_train.npy')
y_test_correct = np.load('../data/y_test.npy')

# Load X
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train range: {y_train_correct.min():.2f} to {y_train_correct.max():.2f}")
print(f"y_test range:  {y_test_correct.min():.2f} to {y_test_correct.max():.2f}")

# Retrain Ridge with best alpha
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

best_model = Ridge(alpha=100.0)
best_model.fit(X_train, y_train_correct)
preds = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test_correct, preds))
r2 = r2_score(y_test_correct, preds)

print(f"\nModel retrained with correct y!")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# Show actual prices
print(f"\nSample Predictions vs Actual:")
print(f"{'Actual':>12} {'Predicted':>12} {'Difference':>12}")
print("-" * 40)
actual_prices = np.expm1(y_test_correct[:5])
predicted_prices = np.expm1(preds[:5])
for actual, pred in zip(actual_prices, predicted_prices):
    diff = abs(actual - pred)
    print(f"${actual:>11,.0f} ${pred:>11,.0f} ${diff:>11,.0f}")

X_train: (1166, 86)
X_test:  (292, 86)
y_train range: 10.46 to 13.53
y_test range:  10.47 to 13.02

Model retrained with correct y!
RMSE: 0.1294
R²:   0.9006

Sample Predictions vs Actual:
      Actual    Predicted   Difference
----------------------------------------
$    190,000 $    222,720 $     32,720
$    100,000 $    109,690 $      9,690
$    115,000 $    111,321 $      3,679
$    159,000 $    164,351 $      5,351
$    315,500 $    321,427 $      5,927


In [20]:
# Save best model
joblib.dump(best_model, '../models/best_model.pkl')
print("Best model saved!")
print("models/best_model.pkl")
print(f"\nFinal Model: Ridge Regression (alpha=100)")
print(f"R² Score: 0.9006")
print(f"RMSE: 0.1294")


Best model saved!
models/best_model.pkl

Final Model: Ridge Regression (alpha=100)
R² Score: 0.9006
RMSE: 0.1294
